In [ ]:
import pandas as pd
import numpy as np

all = pd.read_csv("variant_dataset.csv", low_memory=False)
cancer = pd.read_csv("cancer_dataset.csv", low_memory=False)
pah = pd.read_csv("pah_dataset.csv", low_memory=False)
cftr = pd.read_csv("cftr_dataset.csv", low_memory=False)

In [3]:
print(len(all.columns))
print(len(cancer.columns))
print(len(pah.columns))
print(len(cftr.columns))

77
77
77
77


In [5]:
print(len(all.columns))
print(len(cancer.columns))
print(len(pah.columns))
print(len(cftr.columns))

75
75
75
75


In [6]:
def categorize_mane(row):
    # Satır boş veya NaN ise
    if pd.isna(row) or row.strip() == "":
        return "Empty"

    # Satırı ';' ile böl ve '.' veya boşları temizle
    entries = [x.strip() for x in row.split(";") if x.strip() != "." and x.strip() != ""]

    # . işaretleri temizlendiği için entries listesi boşsa Empty
    if len(entries) == 0:
        return "Empty"

    # Kategorileri kontrol et
    has_select = any(e.lower() == "select" for e in entries)
    has_plus = any(e.lower() == "plus_clinical" for e in entries)

    if has_select and has_plus:
        return "Both"
    elif has_select:
        return "Select"
    elif has_plus:
        return "Plus_Clinical"
    else:
        return "Empty"

In [7]:
all['MANE_category'] = all['MANE'].apply(categorize_mane)
cancer['MANE_category'] = cancer['MANE'].apply(categorize_mane)
pah['MANE_category'] = pah['MANE'].apply(categorize_mane)
cftr['MANE_category'] = cftr['MANE'].apply(categorize_mane)

In [8]:
print(len(all.columns))
print(len(cancer.columns))
print(len(pah.columns))
print(len(cftr.columns))

76
76
76
76


In [9]:
all = pd.get_dummies(all, columns=['MANE_category'], prefix='MANE')
cancer = pd.get_dummies(cancer, columns=['MANE_category'], prefix='MANE')
pah = pd.get_dummies(pah, columns=['MANE_category'], prefix='MANE')
cftr = pd.get_dummies(cftr, columns=['MANE_category'], prefix='MANE')

In [10]:
all = all.drop(columns="MANE")
cancer = cancer.drop(columns="MANE")
pah = pah.drop(columns="MANE")
cftr = cftr.drop(columns="MANE")

In [11]:
print(len(all.columns))
print(len(cancer.columns))
print(len(pah.columns))
print(len(cftr.columns))

78
78
76
76


In [12]:
all = all.drop(columns=['#chr', 'pos(1-based)', 'ref', 'alt'])
cancer = cancer.drop(columns=['#chr', 'pos(1-based)', 'ref', 'alt', 'genename'])
pah = pah.drop(columns=['#chr', 'pos(1-based)', 'ref', 'alt', 'genename'])
cftr = cftr.drop(columns=['#chr', 'pos(1-based)', 'ref', 'alt', 'genename'])

In [13]:
print(len(all.columns))
print(len(cancer.columns))
print(len(pah.columns))
print(len(cftr.columns))

74
73
71
71


In [14]:
def add_missing_flags(df, target_col=None):
    """
    Özniteliklerde was_na sütunu ekler:
    - Eğer hiç NaN yoksa eklemez
    - Multi-transcript features (count varsa) için eklemez
    - Single features ve NaN varsa ekler
    """
    df = df.copy()

    suffixes = ("_max", "_mean", "_std", "_count")
    used_cols = set()
    groups = {}

    # 🔹 1. multi-feature grupları
    for col in df.columns:
        if col == target_col:
            continue

        for suf in suffixes:
            if col.endswith(suf):
                base = col[: -len(suf)]
                groups.setdefault(base, []).append(col)
                used_cols.add(col)
                break

    # 🔹 2. grup flag (multi-feature)
    for base, cols in groups.items():

        # Eğer count sütunu varsa → multi
        count_col = base + "_count"

        # Multi-transcript ve count varsa ekleme yapma
        if count_col in df.columns:
            continue

        # Eğer sütunlarda NaN varsa, flag ekle
        if df[cols].isna().sum().sum() > 0:
            df[base + "_was_na"] = df[cols].isna().any(axis=1).astype(int)

    # 🔹 3. single feature için flag
    for col in df.columns:
        if col == target_col:
            continue
        if col in used_cols:
            continue
        if col.endswith("_was_na"):
            continue

        # numeric feature ve NaN varsa flag ekle
        if df[col].dtype in [np.float64, np.int64]:
            if df[col].isna().sum() > 0:
                df[col + "_was_na"] = df[col].isna().astype(int)

    return df

In [15]:
all = add_missing_flags(all, "label")
cancer = add_missing_flags(cancer, "label")
pah = add_missing_flags(pah, "label")
cftr = add_missing_flags(cftr, "label")

In [16]:
print(len(all.columns))
print(len(cancer.columns))
print(len(pah.columns))
print(len(cftr.columns))

93
90
87
84


In [17]:
from sklearn.model_selection import train_test_split


X_cancer = cancer.drop('label', axis=1)
y_cancer = cancer['label']

X_train_cancer, X_test_cancer, y_train_cancer, y_test_cancer = train_test_split(
    X_cancer,
    y_cancer,
    test_size=0.2,       # Verinin %20'si test, %80'i train olur
    random_state=42,    # Sonuçların her çalıştırdığında aynı çıkması için
    stratify=y_cancer          # Sınıf dengesini koruyan parametre
)


X_pah = pah.drop('label', axis=1)
y_pah = pah['label']

X_train_pah, X_test_pah, y_train_pah, y_test_pah = train_test_split(
    X_pah,
    y_pah,
    test_size=0.2,       # Verinin %20'si test, %80'i train olur
    random_state=42,    # Sonuçların her çalıştırdığında aynı çıkması için
    stratify=y_pah          # Sınıf dengesini koruyan parametre
)


X_cftr = cftr.drop('label', axis=1)
y_cftr = cftr['label']

X_train_cftr, X_test_cftr, y_train_cftr, y_test_cftr = train_test_split(
    X_cftr,
    y_cftr,
    test_size=0.2,       # Verinin %20'si test, %80'i train olur
    random_state=42,    # Sonuçların her çalıştırdığında aynı çıkması için
    stratify=y_cftr         # Sınıf dengesini koruyan parametre
)

In [18]:
cancer_train = X_train_cancer.copy()
cancer_train["label"] = y_train_cancer.copy()

cancer_test = X_test_cancer.copy()
cancer_test["label"] = y_test_cancer.copy()

In [19]:
pah_train = X_train_pah.copy()
pah_train["label"] = y_train_pah.copy()

pah_test = X_test_pah.copy()
pah_test["label"] = y_test_pah.copy()

In [20]:
cftr_train = X_train_cftr.copy()
cftr_train["label"] = y_train_cftr.copy()

cftr_test = X_test_cftr.copy()
cftr_test["label"] = y_test_cftr.copy()

In [21]:
cancer_median = cancer_train.median(numeric_only=True)

cancer_train = cancer_train.fillna(cancer_median)
cancer_test = cancer_test.fillna(cancer_median)

In [22]:
for c in cancer_train.columns:
  if cancer_train[c].isnull().sum() > 0:
    print(c)

In [23]:
pah_median = pah_train.median(numeric_only=True)

pah_train = pah_train.fillna(pah_median)
pah_test = pah_test.fillna(pah_median)

In [24]:
for c in pah_train.columns:
  if pah_train[c].isnull().sum() > 0:
    print(c)

In [25]:
cftr_median = cftr_train.median(numeric_only=True)

cftr_train = cftr_train.fillna(cftr_median)
cftr_test = cftr_test.fillna(cftr_median)

In [26]:
for c in cftr_train.columns:
  if cftr_train[c].isnull().sum() > 0:
    print(c)

In [ ]:
all.to_csv("all_train_test.csv", index=False)

cancer_train.to_csv("cancer_train.csv", index=False)
cancer_test.to_csv("cancer_test.csv", index=False)

pah_train.to_csv("pah_train.csv", index=False)
pah_test.to_csv("pah_test.csv", index=False)

cftr_train.to_csv("cftr_train.csv", index=False)
cftr_test.to_csv("cftr_test.csv", index=False)